In [2]:
import polars as pl
import pandas as pd
from pathlib import Path
from glob import glob

DATA = Path("../data")
OUTPUTS = Path("../outputs")

submission = pd.read_csv("submission.csv")

print("Submission shape:", submission.shape)
submission.head()

Submission shape: (64062, 4)


,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.085805,NaN,NaN
1,ACCT_000007,0.017724,NaN,NaN
2,ACCT_000009,0.022062,NaN,NaN
3,ACCT_000015,0.004340,NaN,NaN
4,ACCT_000016,0.007804,NaN,NaN


In [3]:
HIGH_RISK_THRESHOLD = 0.7

high_risk_accounts = submission[
    submission["is_mule"] > HIGH_RISK_THRESHOLD
]["account_id"]

print("High risk accounts:", len(high_risk_accounts))

High risk accounts: 1699


In [4]:
# Convert to Python set for fast filtering
high_risk_set = set(high_risk_accounts)

txn_files = sorted(glob("../data/transactions/batch-*/part_*.parquet"))

print("Total transaction files:", len(txn_files))

Total transaction files: 396


In [5]:
txn_list = []

for file in txn_files:
    
    df = (
        pl.scan_parquet(file)
        .filter(pl.col("account_id").is_in(high_risk_set))
        .select([
            "account_id",
            "transaction_timestamp",
            "amount",
            "txn_type"
        ])
        .collect()
    )
    
    if df.height > 0:
        txn_list.append(df)

print("Files containing high-risk activity:", len(txn_list))

Files containing high-risk activity: 388


In [6]:
txns = pl.concat(txn_list)

print("Filtered transaction shape:", txns.shape)

txns = txns.with_columns(
    pl.col("transaction_timestamp").str.to_datetime(
        format="%Y-%m-%dT%H:%M:%S",
        strict=False
    ).alias("ts")
)

txns.head()

Filtered transaction shape: (3353438, 4)


account_id,transaction_timestamp,amount,txn_type,ts
str,str,f64,str,datetime[μs]
"""ACCT_000116""","""2020-07-02T12:20:57""",1000.0,"""C""",2020-07-02 12:20:57
"""ACCT_000116""","""2020-07-03T13:46:20""",469.25,"""C""",2020-07-03 13:46:20
"""ACCT_000116""","""2020-07-06T15:13:18""",181.72,"""C""",2020-07-06 15:13:18
"""ACCT_000116""","""2020-07-07T17:46:51""",3191.42,"""D""",2020-07-07 17:46:51
"""ACCT_000116""","""2020-07-09T09:21:23""",43.8,"""D""",2020-07-09 09:21:23


In [7]:
txns = txns.sort(["account_id", "ts"])

print("Sorted transactions ready.")

Sorted transactions ready.


In [9]:
txns = txns.with_columns(
    (
        pl.col("ts")
        .diff()
        .over("account_id")
        .dt.total_seconds()
    ).alias("gap_seconds")
)

txns.head()

account_id,transaction_timestamp,amount,txn_type,ts,gap_seconds
str,str,f64,str,datetime[μs],i64
"""ACCT_000116""","""2020-07-02T12:20:57""",1000.0,"""C""",2020-07-02 12:20:57,null
"""ACCT_000116""","""2020-07-03T13:46:20""",469.25,"""C""",2020-07-03 13:46:20,91523
"""ACCT_000116""","""2020-07-06T15:13:18""",181.72,"""C""",2020-07-06 15:13:18,264418
"""ACCT_000116""","""2020-07-07T17:46:51""",3191.42,"""D""",2020-07-07 17:46:51,95613
"""ACCT_000116""","""2020-07-09T09:21:23""",43.8,"""D""",2020-07-09 09:21:23,142472


In [10]:
SESSION_GAP = 86400  # 24 hours

txns = txns.with_columns(
    (
        (pl.col("gap_seconds") > SESSION_GAP)
        .fill_null(True)
        .cast(pl.Int32)
        .cum_sum()
        .over("account_id")
    ).alias("session_id")
)

In [11]:
txns.select(["account_id","ts","gap_seconds","session_id"]).head(20)

account_id,ts,gap_seconds,session_id
str,datetime[μs],i64,i32
"""ACCT_000116""",2020-07-02 12:20:57,null,1
"""ACCT_000116""",2020-07-03 13:46:20,91523,2
"""ACCT_000116""",2020-07-06 15:13:18,264418,3
"""ACCT_000116""",2020-07-07 17:46:51,95613,4
"""ACCT_000116""",2020-07-09 09:21:23,142472,5
…,…,…,…
"""ACCT_000116""",2020-07-17 21:44:59,107660,9
"""ACCT_000116""",2020-07-19 07:03:08,119889,10
"""ACCT_000116""",2020-07-20 12:32:22,106154,11


In [12]:
sessions = (
    txns
    .group_by(["account_id", "session_id"])
    .agg([
        pl.col("ts").min().alias("start"),
        pl.col("ts").max().alias("end"),
        pl.len().alias("txn_count"),
        pl.col("amount").sum().alias("total_amount"),
        (pl.col("txn_type") == "C").sum().alias("credits"),
        (pl.col("txn_type") == "D").sum().alias("debits")
    ])
)

sessions.head()

account_id,session_id,start,end,txn_count,total_amount,credits,debits
str,i32,datetime[μs],datetime[μs],u32,f64,u32,u32
"""ACCT_173209""",87,2023-05-21 06:49:18,2023-05-21 09:18:02,2,4500.0,0,2
"""ACCT_085868""",133,2024-12-25 14:29:01,2024-12-25 14:29:01,1,15000.0,0,1
"""ACCT_160668""",106,2021-11-30 06:56:15,2021-12-09 13:26:02,19,251589.02,7,12
"""ACCT_014842""",184,2023-08-07 11:48:16,2023-08-07 11:53:54,2,1722.86,1,1
"""ACCT_160872""",47,2023-10-22 18:01:33,2023-10-26 20:56:10,18,194852.59,8,10


In [13]:
sessions = sessions.with_columns(
    (
        pl.col("txn_count") * 0.5 +
        pl.col("total_amount").abs() * 0.00001 +
        pl.col("credits") * 0.3 +
        pl.col("debits") * 0.3
    ).alias("suspicion_score")
)

sessions.select(
    ["account_id","session_id","txn_count","total_amount","suspicion_score"]
).head()

account_id,session_id,txn_count,total_amount,suspicion_score
str,i32,u32,f64,f64
"""ACCT_173209""",87,2,4500.0,1.645
"""ACCT_085868""",133,1,15000.0,0.95
"""ACCT_160668""",106,19,251589.02,17.71589
"""ACCT_014842""",184,2,1722.86,1.6172286
"""ACCT_160872""",47,18,194852.59,16.348526


In [14]:
best_windows = (
    sessions
    .sort("suspicion_score", descending=True)
    .group_by("account_id")
    .head(1)
)

best_windows.head()

account_id,session_id,start,end,txn_count,total_amount,credits,debits,suspicion_score
str,i32,datetime[μs],datetime[μs],u32,f64,u32,u32,f64
"""ACCT_186486""",228,2025-03-16 16:45:21,2025-03-22 11:05:28,19,5.0277e6,9,10,65.477137
"""ACCT_177349""",60,2025-05-21 17:56:32,2025-05-30 21:50:00,42,2.1761e6,18,24,55.360613
"""ACCT_070041""",24,2020-09-05 03:52:26,2020-09-05 23:03:28,7,2.2064e6,2,5,27.66354
"""ACCT_154037""",561,2025-05-19 10:14:55,2025-05-22 16:58:42,9,7.3702e6,3,6,80.901571
"""ACCT_069341""",4,2024-11-24 18:00:24,2025-03-02 00:04:58,692,1.4272e7,337,355,696.321515


In [15]:
best_pd = best_windows.select([
    "account_id",
    "start",
    "end"
]).to_pandas()

submission = submission.merge(
    best_pd,
    on="account_id",
    how="left"
)

submission["suspicious_start"] = submission["start"]
submission["suspicious_end"] = submission["end"]

submission = submission.drop(columns=["start","end"])

submission.head()

,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.085805,NaT,NaT
1,ACCT_000007,0.017724,NaT,NaT
2,ACCT_000009,0.022062,NaT,NaT
3,ACCT_000015,0.004340,NaT,NaT
4,ACCT_000016,0.007804,NaT,NaT


In [16]:
submission[submission["suspicious_start"].notna()].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.943777,2021-06-06 16:15:02,2021-06-17 07:54:55
92,ACCT_000236,0.925378,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.990282,2025-05-11 16:41:00,2025-05-29 21:22:39
177,ACCT_000484,0.855262,2024-02-12 10:02:06,2024-03-01 15:56:31
231,ACCT_000646,0.938796,2024-05-13 21:31:14,2024-05-17 14:41:56


In [17]:
submission.to_csv("submission_with_temporal_windows.csv", index=False)

print("Final submission saved.")

Final submission saved.


In [18]:
submission["suspicious_start"].notna().sum()

np.int64(1613)

In [19]:
submission[
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].notna())
].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.943777,2021-06-06 16:15:02,2021-06-17 07:54:55
92,ACCT_000236,0.925378,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.990282,2025-05-11 16:41:00,2025-05-29 21:22:39
177,ACCT_000484,0.855262,2024-02-12 10:02:06,2024-03-01 15:56:31
231,ACCT_000646,0.938796,2024-05-13 21:31:14,2024-05-17 14:41:56


In [20]:
submission[
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].isna())
].head()

,account_id,is_mule,suspicious_start,suspicious_end
856,ACCT_002584,0.876619,NaT,NaT
1070,ACCT_003281,0.974394,NaT,NaT
1590,ACCT_004974,0.988538,NaT,NaT
1608,ACCT_005047,0.811200,NaT,NaT
2371,ACCT_007358,0.911922,NaT,NaT


In [119]:
txn_basic = pl.read_parquet("../outputs/features_txn_basic.parquet")

fallback_windows = (
    txn_basic
    .select([
        "account_id",
        pl.col("first_txn_date").alias("fallback_start"),
        pl.col("last_txn_date").alias("fallback_end")
    ])
    .to_pandas()
)

submission = submission.merge(
    fallback_windows,
    on="account_id",
    how="left"
)

mask = (
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].isna())
)

submission.loc[mask, "suspicious_start"] = submission.loc[mask, "fallback_start"]
submission.loc[mask, "suspicious_end"] = submission.loc[mask, "fallback_end"]

submission = submission.drop(columns=["fallback_start","fallback_end"])

In [103]:
submission["suspicious_start"].notna().sum()

np.int64(1613)

In [124]:
high_risk = submission[submission["is_mule"] > 0.7]["account_id"]

txn_basic = pl.read_parquet("../outputs/features_txn_basic.parquet")

accounts_with_txns = set(txn_basic["account_id"].to_list())
high_risk_set = set(high_risk)

missing_txn_accounts = high_risk_set - accounts_with_txns

print("High risk accounts:", len(high_risk_set))
print("High risk with transactions:", len(high_risk_set & accounts_with_txns))
print("High risk WITHOUT transactions:", len(missing_txn_accounts))

High risk accounts: 1415
High risk with transactions: 1415
High risk WITHOUT transactions: 0


In [104]:
submission[submission["suspicious_start"].notna()].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.562036,2021-06-06T16:15:02,2021-06-17T07:54:55
92,ACCT_000236,0.548913,2025-06-04T17:35:48,2025-06-07T10:01:13
161,ACCT_000440,0.595431,2025-05-11T16:41:00,2025-05-29T21:22:39
177,ACCT_000484,0.499389,2024-02-12T10:02:06,2024-03-01T15:56:31
231,ACCT_000646,0.558478,2024-05-13T21:31:14,2024-05-17T14:41:56


In [105]:
best_windows.head()
print(best_windows.shape)

(1699, 9)


In [106]:
submission.columns

Index(['account_id', 'is_mule', 'suspicious_start', 'suspicious_end'], dtype='str')

In [107]:
print(submission["account_id"].dtype)
print(best_windows["account_id"].dtype)

str
String


In [108]:
best_pd = best_windows.select([
    "account_id",
    "start",
    "end"
]).to_pandas()

best_pd["account_id"] = best_pd["account_id"].astype(str)
submission["account_id"] = submission["account_id"].astype(str)

submission = submission.merge(
    best_pd,
    on="account_id",
    how="left"
)

submission["suspicious_start"] = submission["start"]
submission["suspicious_end"] = submission["end"]

submission = submission.drop(columns=["start","end"])

In [109]:
submission["suspicious_start"].notna().sum()

np.int64(1613)

In [112]:
submission.isnull().sum()

account_id              0
is_mule                 0
suspicious_start    62449
suspicious_end      62449
dtype: int64

In [113]:
submission["suspicious_start"] = submission["suspicious_start"].astype(str).str.replace(" ", "T")
submission["suspicious_end"] = submission["suspicious_end"].astype(str).str.replace(" ", "T")

In [114]:
submission.to_csv("submission_final.csv", index=False)

In [118]:
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])

print("Max probability:", submission["is_mule"].max())

print("Accounts >0.5:", (submission["is_mule"]>0.5).sum())
print("Accounts >0.7:", (submission["is_mule"]>0.7).sum())

print("Temporal windows:", submission["suspicious_start"].notna().sum())

Rows: 64062
Columns: 4
Max probability: 0.6325717364016288
Accounts >0.5: 1099
Accounts >0.7: 0
Temporal windows: 1613


In [120]:
max_p = submission["is_mule"].max()
min_p = submission["is_mule"].min()

submission["is_mule"] = (submission["is_mule"] - min_p) / (max_p - min_p)

In [121]:
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])

print("Max probability:", submission["is_mule"].max())

print("Accounts >0.3:", (submission["is_mule"]>0.3).sum())
print("Accounts >0.5:", (submission["is_mule"]>0.5).sum())
print("Accounts >0.7:", (submission["is_mule"]>0.7).sum())

print("Temporal windows:", submission["suspicious_start"].notna().sum())

Rows: 64062
Columns: 4
Max probability: 1.0
Accounts >0.3: 2787
Accounts >0.5: 2074
Accounts >0.7: 1415
Temporal windows: 1613


In [122]:
submission.to_csv("submission_final.csv", index=False)

In [123]:
submission["suspicious_start"].notna().sum()

np.int64(1613)